# Volume + particles: composition through depth peeling

A simulation in the spirit of
[issue #277](https://github.com/K3D-tools/K3D-jupyter/issues/277): a particle
cloud embedded in a volumetric plasma wind around a star. A rotating source
feeds dye into a spiral wind field with curl-noise turbulence; the dye is
advected semi-Lagrangian style on a 160^3 grid for 320 steps (~1-2 minutes in
pure numpy) and rendered as a `volume`, with tracer particles advected by the
same field.


In [1]:
import numpy as np

import k3d

In [2]:
from tqdm.auto import tqdm

rng = np.random.default_rng(277)
n = 160
L = 2.5
dt = 1.0
steps = 320

g = np.linspace(-L, L, n, dtype=np.float32)
dg = g[1] - g[0]
Zg, Yg, Xg = np.meshgrid(g, g, g, indexing='ij')
r_cyl = np.sqrt(Xg ** 2 + Yg ** 2) + 1e-6

# velocity field: a spiral wind plus divergence-free curl-noise turbulence
def smooth(a, passes):
    for _ in range(passes):
        a = sum(np.roll(a, s, ax) for ax in range(3) for s in (-1, 1)) / 6.0
    return a

v_r = 0.0065 * np.clip(r_cyl / 0.35, 0.0, 1.0)          # radial wind (starts past the star)
omega = 0.05 / (0.25 + r_cyl)                             # differential rotation
vx_w = v_r * Xg / r_cyl - omega * Yg
vy_w = v_r * Yg / r_cyl + omega * Xg

A = [smooth(rng.random((n, n, n)).astype(np.float32) - 0.5, 8) for _ in range(3)]
curl_x = (np.roll(A[2], -1, 1) - np.roll(A[2], 1, 1)) - (np.roll(A[1], -1, 0) - np.roll(A[1], 1, 0))
curl_y = (np.roll(A[0], -1, 0) - np.roll(A[0], 1, 0)) - (np.roll(A[2], -1, 2) - np.roll(A[2], 1, 2))
curl_z = (np.roll(A[1], -1, 2) - np.roll(A[1], 1, 2)) - (np.roll(A[0], -1, 1) - np.roll(A[0], 1, 1))
amp = 0.55 * np.clip(r_cyl / 0.6, 0.0, 1.0)
vx = vx_w + amp * curl_x
vy = vy_w + amp * curl_y
vz = amp * curl_z * 0.4

# semi-Lagrangian dye advection (grid indexed [z, y, x])
def trilinear(field, fi, fj, fk):
    i0 = np.floor(fi).astype(np.int32); j0 = np.floor(fj).astype(np.int32); k0 = np.floor(fk).astype(np.int32)
    di = (fi - i0).astype(np.float32); dj = (fj - j0).astype(np.float32); dk = (fk - k0).astype(np.float32)
    i0 = np.clip(i0, 0, n - 2); j0 = np.clip(j0, 0, n - 2); k0 = np.clip(k0, 0, n - 2)
    i1 = i0 + 1; j1 = j0 + 1; k1 = k0 + 1
    return (field[i0, j0, k0] * (1 - di) * (1 - dj) * (1 - dk)
            + field[i1, j0, k0] * di * (1 - dj) * (1 - dk)
            + field[i0, j1, k0] * (1 - di) * dj * (1 - dk)
            + field[i0, j0, k1] * (1 - di) * (1 - dj) * dk
            + field[i1, j1, k0] * di * dj * (1 - dk)
            + field[i1, j0, k1] * di * (1 - dj) * dk
            + field[i0, j1, k1] * (1 - di) * dj * dk
            + field[i1, j1, k1] * di * dj * dk)

I, J, K = np.meshgrid(np.arange(n, dtype=np.float32), np.arange(n, dtype=np.float32),
                      np.arange(n, dtype=np.float32), indexing='ij')
bi = I - vz * dt / dg
bj = J - vy * dt / dg
bk = K - vx * dt / dg

# source: a thin equatorial ring near the star, modulated by two rotating arms
theta = np.arctan2(Yg, Xg)
source = (np.exp(-((r_cyl - 0.4) / 0.045) ** 2) * np.exp(-(Zg / 0.04) ** 2)).astype(np.float32)

dye = np.zeros((n, n, n), dtype=np.float32)
for s in tqdm(range(steps), desc='advecting dye'):
    dye = trilinear(dye, bi, bj, bk)
    mod = (0.15 + 0.85 * (0.5 + 0.5 * np.cos(2.0 * theta + 0.06 * s)) ** 2).astype(np.float32)
    dye += source * mod * 4.0
    dye *= 0.997

dye *= (0.25 + r_cyl) ** 1.5  # compensate the radial dilution of the wind
density = (dye * 1300.0 / np.percentile(dye[dye > 0.01], 99.7)).astype(np.float32)

advecting dye:   0%|          | 0/320 [00:00<?, ?it/s]

Without depth peeling the volume cannot interleave with geometry - the
particles render entirely in front of (or behind) the plasma:

In [4]:
plot = k3d.plot(grid_visible=False, camera_auto_fit=False)
plot += k3d.volume(
    density, samples=512, alpha_coef=90,
    color_map=k3d.matplotlib_color_maps.jet, color_range=[60, 1300],
    bounds=[-L, L, -L, L, -L, L],
)
plot += k3d.points(np.array([[0.0, 0.0, 0.0]], dtype=np.float32),
                   shader='mesh', point_size=0.5, color=0x2040FF, mesh_detail=4)
plot.camera = [3.6, -3.6, 2.4, 0, 0, 0, 0, 0, 1]
plot.display()

Output()

With `depth_peels >= 3` the ray march splits into segments bounded by the
peel layers and the particles occlude and are occluded sample-accurately
(fewer layers make the segmentation too coarse to be predictable):

In [5]:
plot.depth_peels = 4

The composition is orthogonal to lighting - the advanced renderer adds
environment light and ambient occlusion on top:

In [6]:
plot.renderer = 'advanced'
plot.environment = 'moonless_golf'